# Imports

In [2]:
import requests
import pandas as pd
import os
from dotenv import load_dotenv
import xml.etree.ElementTree as ET #Built-in python library used for working with 
from datetime import datetime, timedelta


# Testing NVE API
This will be used instead of the pd.read in my data_loader.py file

In [3]:
url = "https://biapi.nve.no/magasinstatistikk/api/Magasinstatistikk/HentOffentligData"

response = requests.get(url, timeout=60)
response.raise_for_status()

data = response.json()
df = pd.DataFrame(data)

df.head()
print(df.shape)
print(df.columns.tolist())
print(df.dtypes)

(14913, 11)
['dato_Id', 'omrType', 'omrnr', 'iso_aar', 'iso_uke', 'fyllingsgrad', 'kapasitet_TWh', 'fylling_TWh', 'neste_Publiseringsdato', 'fyllingsgrad_forrige_uke', 'endring_fyllingsgrad']
dato_Id                         str
omrType                         str
omrnr                         int64
iso_aar                       int64
iso_uke                       int64
fyllingsgrad                float64
kapasitet_TWh               float64
fylling_TWh                 float64
neste_Publiseringsdato          str
fyllingsgrad_forrige_uke    float64
endring_fyllingsgrad        float64
dtype: object


# Testing ENTSO-E API

In [4]:
load_dotenv()

url = "https://web-api.tp.entsoe.eu/api"

#All the parameters needed to check API connection works, fetched from the ENTSOE API documentation
params = {
    "securityToken": os.getenv("ENTSOE_API_KEY"),
    "documentType": "A11",
    "out_Domain": "10YNO-2--------T",
    "in_Domain": "10YDK-1--------W",
    "periodStart": "202610010000",
    "periodEnd": "202610020000"
}

response = requests.get(url, params=params, timeout=60)

print("Status:", response.status_code)
print(response.text[:1500])

Status: 200
<?xml version="1.0" encoding="utf-8"?>
<Publication_MarketDocument xmlns="urn:iec62325.351:tc57wg16:451-3:publicationdocument:7:0">
  <mRID>9d0f713acfeb404289f7379d648d1a50</mRID>
  <revisionNumber>1</revisionNumber>
  <type>A11</type>
  <sender_MarketParticipant.mRID codingScheme="A01">10X1001A1001A450</sender_MarketParticipant.mRID>
  <sender_MarketParticipant.marketRole.type>A32</sender_MarketParticipant.marketRole.type>
  <receiver_MarketParticipant.mRID codingScheme="A01">10X1001A1001A450</receiver_MarketParticipant.mRID>
  <receiver_MarketParticipant.marketRole.type>A33</receiver_MarketParticipant.marketRole.type>
  <createdDateTime>2026-10-09T07:34:21Z</createdDateTime>
  <period.timeInterval>
    <start>2026-10-01T00:00Z</start>
    <end>2026-10-02T00:00Z</end>
  </period.timeInterval>
    <TimeSeries>
      <mRID>1</mRID>
      <businessType>A66</businessType>
      <in_Domain.mRID codingScheme="A01">10YDK-1--------W</in_Domain.mRID>
      <out_Domain.mRID codingSc

# XML Extraction
The NVE API returns JSON format, while the ENTSO-E returns in XML, so we would need to extract the relevant data from XML.

In [5]:
#response.content contains the raw XML data we fetched from ENTSO-E API. fromstring converts the raw data into something Python can work with.
root = ET.fromstring(response.content)

#Creates a dictionary by EXtracting the namespace from the root tag. This creates the alias "ns" which provides the dictionary with 
# the actual namespace.
ns = {"ns": root.tag.split("}")[0].strip("{")}

#"ns.:TimeSeries" specifies the element we are looking for using the alias ns, and "ns" provides the dictionary which translates
# the alias into the actual namespace.
series = root.find("ns:TimeSeries", ns)

print("Time series found:", series is not None)

"""Parameters used here:
    Start: start time in YYYY-MM-DDTHH:MMZ format
    End: end time in YYYY-MM-DDTHH:MMZ format
    Resolution: time periode of the data: Here it is PT15M, which means 15 minutes
    Quantity: Average electricity flow in MW for the given resolution.
"""
if series is not None:
    period = series.find("ns:Period", ns)
    
    print("Start:", period.findtext("ns:timeInterval/ns:start", namespaces=ns))
    print("End:", period.findtext("ns:timeInHterval/ns:end", namespaces=ns))
    print("Resolution:", period.findtext("ns:resolution", namespaces=ns))

    for point in period.findall("ns:Point", ns)[:5]:
        print(
            "Position:", point.findtext("ns:position", namespaces=ns),
            "Quantity:", point.findtext("ns:quantity", namespaces=ns)
        )

Time series found: True
Start: 2026-10-01T00:00Z
End: None
Resolution: PT15M
Position: 1 Quantity: 605.97
Position: 2 Quantity: 647.22
Position: 3 Quantity: 653.14
Position: 4 Quantity: 582.93
Position: 5 Quantity: 452.13


# Extracting XML to Pandas dataframe
Now we are extracting the XML to a pandas dataframe, which we will later put in to cassandra using spark. For now Ive just used a small period of one day, and only NO-2 to DK-1. Next step will be to do the same for the whole two year period, and for whole of Norway.

In [6]:
root = ET.fromstring(response.content)
ns = {"ns": root.tag.split("}")[0].strip("{")}

records = []

for series in root.findall("ns:TimeSeries", ns):
    for period in series.findall("ns:Period", ns):

        #Extract start time. timeInterval/ns:start tells python to navigate two levels down from period to find start time.
        #We get a string containing a "Z" which just represents UTC time, so replace it with +00:00 to make it valid string for
        #datetime.fromisoformat.
        start = datetime.fromisoformat(
            period.findtext(
                "ns:timeInterval/ns:start",
                namespaces=ns
            ).replace("Z", "+00:00")
        )

        resolution = period.findtext("ns:resolution", namespaces=ns)
        if resolution != "PT15M":
            raise ValueError(f"Unexpected resolution: {resolution}")
        interval = timedelta(minutes=15)

        for point in period.findall("ns:Point", ns):

            position = int(point.findtext("ns:position", namespaces=ns))
            quantity = float(point.findtext("ns:quantity", namespaces=ns))

            timestamp = start + (position - 1) * interval
            energy_MWh = quantity / 4 # Convert MW to MWh for 15-minute intervals

            records.append({
                "timestamp": timestamp,
                "from_area": "NO2",
                "to_area": "DK1",
                "power_MW": quantity,
                "energy_MWh": energy_MWh
            })

df = pd.DataFrame(records)

display(df.head())
print("Number of observations:", len(df))

,timestamp,from_area,to_area,power_MW,energy_MWh
0,2026-10-01 00:00:00+00:00,NO2,DK1,605.97,151.4925
1,2026-10-01 00:15:00+00:00,NO2,DK1,647.22,161.8050
2,2026-10-01 00:30:00+00:00,NO2,DK1,653.14,163.2850
3,2026-10-01 00:45:00+00:00,NO2,DK1,582.93,145.7325
4,2026-10-01 01:00:00+00:00,NO2,DK1,452.13,113.0325


Number of observations: 96


# API Retrieving for two years

First combining the code written earlier with some new code for the timestamps, to check if we still can retrieve the data. Decided to split the dataset into 4 API calls, because max length per API call was 1 year. Took 6 months instead just to be sure we account for leap year etc.

In [ ]:
load_dotenv()

url = "https://web-api.tp.entsoe.eu/api"

#All the parameters needed to check API connection works, fetched from the ENTSOE API documentation.
#Here periodStart and periodEnd are left empty and they will be filled in the loop below.
params = {
    "securityToken": os.getenv("ENTSOE_API_KEY"),
    "documentType": "A11",
    "out_Domain": "10YNO-2--------T",
    "in_Domain": "10YDK-1--------W",
    "periodStart": "",
    "periodEnd": ""
}

start_date = pd.Timestamp("2024-10-09", tz="UTC")
end_date = pd.Timestamp("2026-10-09", tz="UTC")

date_ranges = []

current = start_date

while current < end_date:
    #Calculate the next date, which is either 6 months from the current date or the end date, whichever is earlier.
    next_date = min(current + pd.DateOffset(months=6), end_date)

    date_ranges.append((current, next_date))

    current = next_date

responses = []

for start, end in date_ranges:
    #Convert timestamps in to correct format
    params["periodStart"] = start.strftime("%Y%m%d%H%M")
    params["periodEnd"] = end.strftime("%Y%m%d%H%M")

    response = requests.get(url, params=params, timeout=60)
    #Checks for HTTP errors, if this is not included the code will continue to run even if the request fails.
    response.raise_for_status()
    
    responses.append(response.content)

    print(f"Retrieved data from {start.date()} to {end.date()}")

Retrieved data from 2024-10-09 to 2025-04-09
Retrieved data from 2025-04-09 to 2025-10-09
Retrieved data from 2025-10-09 to 2026-04-09
Retrieved data from 2026-04-09 to 2026-10-09


# Connecting it together
Using the same XML to Pandas extraction as earlier, but now looping through the 4 XML documents we have.

In [25]:
records = []

for response in responses:
    root = ET.fromstring(response)
    ns = {"ns": root.tag.split("}")[0].strip("{")}

    for series in root.findall("ns:TimeSeries", ns):
        for period in series.findall("ns:Period", ns):

            #Extract start time. timeInterval/ns:start tells python to navigate two levels down from period to find start time.
            #We get a string containing a "Z" which just represents UTC time, so replace it with +00:00 to make it valid string for
            #datetime.fromisoformat.
            start = datetime.fromisoformat(
                period.findtext(
                    "ns:timeInterval/ns:start",
                    namespaces=ns
                ).replace("Z", "+00:00")
            )

            resolution = period.findtext("ns:resolution", namespaces=ns)
            if resolution == "PT15M":
                interval = timedelta(minutes=15)
            elif resolution == "PT60M":
                interval = timedelta(minutes=60)
            else:
                raise ValueError(f"Unexpected resolution: {resolution}")

            for point in period.findall("ns:Point", ns):

                position = int(point.findtext("ns:position", namespaces=ns))
                quantity = float(point.findtext("ns:quantity", namespaces=ns))

                timestamp = start + (position - 1) * interval
                energy_MWh = quantity * (interval.total_seconds()/3600)  # Convert MW to MWh depending on interval length

                records.append({
                    "timestamp": timestamp,
                    "from_area": "NO2",
                    "to_area": "DK1",
                    "power_MW": quantity,
                    "energy_MWh": energy_MWh,
                    "interval_minutes": int(interval.total_seconds() / 60)
                })

df = pd.DataFrame(records)

df = df.sort_values("timestamp").reset_index(drop=True)

display(df.head())
display(df.tail())

print("Number of observations:", len(df))
print("First timestamp:", df["timestamp"].min())
print("Last timestamp:", df["timestamp"].max())
print("Duplicate timestamps:", df["timestamp"].duplicated().sum())
print(df["interval_minutes"].value_counts())

print("Total hours covered:",
      df["interval_minutes"].sum() / 60)

,timestamp,from_area,to_area,power_MW,energy_MWh,interval_minutes
0,2024-10-09 00:00:00+00:00,NO2,DK1,701.10,701.10,60
1,2024-10-09 01:00:00+00:00,NO2,DK1,615.15,615.15,60
2,2024-10-09 02:00:00+00:00,NO2,DK1,698.33,698.33,60
3,2024-10-09 03:00:00+00:00,NO2,DK1,696.65,696.65,60
4,2024-10-09 04:00:00+00:00,NO2,DK1,811.06,811.06,60


,timestamp,from_area,to_area,power_MW,energy_MWh,interval_minutes
40824,2026-10-08 09:00:00+00:00,NO2,DK1,0.94,0.235,15
40825,2026-10-08 09:15:00+00:00,NO2,DK1,2.06,0.515,15
40826,2026-10-08 09:30:00+00:00,NO2,DK1,0.00,0.000,15
40827,2026-10-08 15:45:00+00:00,NO2,DK1,0.06,0.015,15
40828,2026-10-08 16:00:00+00:00,NO2,DK1,0.00,0.000,15


Number of observations: 40829
First timestamp: 2024-10-09 00:00:00+00:00
Last timestamp: 2026-10-08 16:00:00+00:00
Duplicate timestamps: 0
interval_minutes
15    36899
60     3930
Name: count, dtype: int64
Total hours covered: 13154.75


# Transfer Data Function
We have now looked at data for two  years, but only in the areas from NO2 to DK1. We would want to get the data for the whole of the 5 Norwegian areas, both import and export. 
Therefore the next step is to make a function which:
- Generates six-month date ranges up to two years,
- Request data from ENTSO-E API for each date
- Parse the XML responses
- Convert these responses into a pd dataframe

In [ ]:
def get_transfer_data(out_domain, in_domain, start_date, end_date):
    """
    Fetches electricity transfer data from the ENTSOE API for the specified domains and date range.

    Parameters:
    - out_domain (str): Exporting domain
    - in_domain (str): Importing domain.
    - start_date (str): Start date
    - end_date (str): End date

    Returns:
    - Relevant parameters used to extract from the API
    """
    
    load_dotenv()

    url = "https://web-api.tp.entsoe.eu/api"
    start_date = pd.Timestamp(start_date, tz="UTC")
    end_date = pd.Timestamp(end_date, tz="UTC")

    #All the parameters needed to check API connection works, fetched from the ENTSOE API documentation.
    #Here periodStart and periodEnd are left empty and they will be filled in the loop below.
    params = {
        "securityToken": os.getenv("ENTSOE_API_KEY"),
        "documentType": "A11",
        "out_Domain": out_domain,
        "in_Domain": in_domain,
        "periodStart" : start_date.strftime("%Y%m%d%H%M"),
        "periodEnd": end_date.strftime("%Y%m%d%H%M")
    }

    date_ranges = []

    current = params["periodStart"]

    while current < end_date:
        #Calculate the next date, which is either 6 months from the current date or the end date, whichever is earlier.
        next_date = min(current + pd.DateOffset(months=6), params["periodEnd"])

        date_ranges.append((current, next_date))

        current = next_date

    responses = []

    for start, end in date_ranges:

        response = requests.get(url, params=params, timeout=60)
        #Checks for HTTP errors, if this is not included the code will continue to run even if the request fails.
        response.raise_for_status()
        
        responses.append(response.content)

        print(f"Retrieved data from {start.date()} to {end.date()}")

    records = []

    for response in responses:
        root = ET.fromstring(response)
        ns = {"ns": root.tag.split("}")[0].strip("{")}

        for series in root.findall("ns:TimeSeries", ns):
            for period in series.findall("ns:Period", ns):

                #Extract start time. timeInterval/ns:start tells python to navigate two levels down from period to find start time.
                #We get a string containing a "Z" which just represents UTC time, so replace it with +00:00 to make it valid string for
                #datetime.fromisoformat.
                start = datetime.fromisoformat(
                    period.findtext(
                        "ns:timeInterval/ns:start",
                        namespaces=ns
                    ).replace("Z", "+00:00")
                )

                resolution = period.findtext("ns:resolution", namespaces=ns)
                if resolution == "PT15M":
                    interval = timedelta(minutes=15)
                elif resolution == "PT60M":
                    interval = timedelta(minutes=60)
                else:
                    raise ValueError(f"Unexpected resolution: {resolution}")

                for point in period.findall("ns:Point", ns):

                    position = int(point.findtext("ns:position", namespaces=ns))
                    quantity = float(point.findtext("ns:quantity", namespaces=ns))

                    timestamp = start + (position - 1) * interval
                    energy_MWh = quantity * (interval.total_seconds()/3600)  # Convert MW to MWh depending on interval length

                    records.append({
                        "timestamp": timestamp,
                        "from_area": "NO2",
                        "to_area": "DK1",
                        "power_MW": quantity,
                        "energy_MWh": energy_MWh,
                    })

    df = pd.DataFrame(records)
    return df

# AI Usage

# Log

Given the resolution(time interval) is 15 minutes, we need to do quanity * 0,25 to get the MWh hourly rate.
Some missing data, meaning not all total hours are covered